In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim

from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# ミッション5

下記の命令を組み合わせてプログラムを書き、ロボットをコースに沿って移動させ、コース外周に設けられた標識をみつけたらコースから外れて標識の直前(0.5m手前)で停車しよう。

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置を把握する
3. ２つ下のセル内において
    - プログラムを書く
    - 路面標識を好きな位置に設置する
    - 実行して結果を見る
4. シミュレータでうまく動いたらロボットにプログラムを書きこんで動かしてみよう（講師に声をかけてね）

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|rotate|一定速度で回転する|w=回転速度[度/秒]|rotate(w=90)|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = Search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|
|auto|コースに沿って進む|v=速度[m/秒]|auto(v=0.2)|

## 注意点
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大5度ほどずれる

In [28]:
class Mission5Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2, 2), 0.5, should_stop=True),
        ]
        self.initial_xy = (2.5, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=2, y=2, name="original"),
            ]
        )

print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Mission5Base()).show()


最大速度 0.22 m/秒
最大回転速度 162.72 度/秒


In [ ]:
class Mission5(Mission5Base):
    @staticmethod
    def command_func(*, move, rotate, search, auto, **kwargs):
        # ヒントとして、「searchして標識が見つからなかったらコースに沿って進む」を繰り返すという処理を記載済み
        # 「searchして標識が見つかった場合」のプログラムだけを書けばOK

        while True:
            pos = search()
            if pos is None:
                auto(v=0.2)
            else:
                ######## ここから下にプログラムを書こう
                move(v=0.0) # 書き方の例
                ######## ここより上にプログラムを書こう


sim = CarSim(prop, Mission5())
sim.run()
SimDrawer(sim).show()

drive_dt=0.031, detect_dt=0.061, throttle=20
[26.026] all goals reached
[26.026] simulation_func finished
    takes 1.304s
    ideal 1.301s
Trajectory points : 848


100%|██████████| 284/284 [00:01<00:00, 235.86it/s]


# ヒント

- コースの外にでるには、autoでなくmoveを使う必要がある
- 標識を見つけたら、「moveやrotateを使用して標識の方に向かう」というプログラムを書こう
- ただし、それだけだと「標識の直前(0.5m手前)で停車」という動作ができないので・・・
    - 「標識の直前(0.5m手前)になったら停止する」というプログラムを書けばよい
    - 条件分岐を使って書いてみよう
    - 標識までの前方距離はpos.x、標識までの距離はpos.rなのでこのどちらかを使おう